# Spark Dataframe

## 1. Connection

In [ ]:
from faker import Faker
from pyspark.sql import SparkSession

Spark Connect is a client-server architecture within Apache Spark that enables remote connectivity to Spark clusters from any application. PySpark provides the client for the Spark Connect server, allowing Spark to be used as a service.

Pyspark also needs jvm, spark master and workers run their own jvm

`SparkSession.builder.master()`
This is the classic Spark API. It tells the Spark driver where to submit jobs.
The driver (your Python process) runs on your machine, connects to the Spark Master, and schedules tasks on the workers.


`SparkSession.builder.remote()` is newer and is used with Spark Connect.
Instead of embedding the Spark driver inside your Python process, your Python code becomes a thin client. The actual Spark driver runs remotely in a Spark Connect server.


Python in worker has different version: 3.10 than that in driver: 3.12, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set
Pyspark driver python version and spark worker python version should be same

In [ ]:
# test that connection works

spark = (
    SparkSession.builder
    .master("spark://localhost:7077")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9100")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .appName("pyspark-labs")
    .getOrCreate()
)

print(spark.version)
spark.range(5).show()

## 2. Creating dataframe

In [ ]:
# Dataframe
from pyspark.sql import Row
from random import randint

fake = Faker()

In [ ]:
def generate_user_data(n):
    """Generate and return user data one at a time."""
    for _ in range(n):
        yield {
            "name": fake.name(),
            "email": fake.email(),
            "phone_number": fake.phone_number(),
            "job": fake.job(),
            "date_of_birth": fake.date_of_birth(),
            "company": fake.company(),
            "address": fake.address(),
            "ssn": fake.ssn(),
            "state": fake.state(),
            "country": fake.country(),
            "salary": randint(50_000, 200_000)
        }

In [ ]:

n = 100

df = spark.createDataFrame([
    Row(name=user["name"],
        email=user["email"],
        phone_number=user["phone_number"],
        job=user["job"],
        date_of_birth=user["date_of_birth"],
        company=user["company"],
        address=user["address"],
        ssn=user["ssn"],
        state=user["state"],
        country=user["country"],
        salary=user["salary"]
    )
    for user in generate_user_data(n)
])
df.show()

In [ ]:
# Create a PySpark DataFrame with an explicit schema.
n = 100

df = spark.createDataFrame(
    [
        Row(
            name=user["name"],
            email=user["email"],
            phone_number=user["phone_number"],
            job=user["job"],
            date_of_birth=user["date_of_birth"],
            company=user["company"],
            address=user["address"],
            ssn=user["ssn"],
            state=user["state"],
            country=user["country"],
            salary=user["salary"]
        ) for user in generate_user_data(n)
    ],
    schema="name string, email string, phone_number string, job string, date_of_birth date, company string, address string, ssn string, state string, country string, salary int"
)
# string, int, date, timestamp, boolean
df.show()


In [ ]:
# Create a PySpark DataFrame from a pandas DataFrame
import pandas as pd

In [ ]:
pdf = pd.DataFrame([user for user in generate_user_data(n)])
print(pdf.shape)
df = spark.createDataFrame(pdf)
df.show()

## 3. Viewing data

In [ ]:
# All DataFrames above result same.
df.show()
df.printSchema()

In [ ]:
# The top rows of a DataFrame can be displayed using DataFrame.show().

df.show(1, truncate=False)  # Show the first row without truncating the columns.
df.show(1, truncate=False, vertical=True)  # Show the first row vertically.

In [ ]:
# schema
print(df.columns)
df.printSchema()

In [ ]:
# summary of dataframe
df.select("name", "email", "date_of_birth").describe().show(truncate=False)

In [ ]:
df.select("name", "email", "date_of_birth").summary().show(truncate=False) # can also use summary instead of describe

In [ ]:
# collect, take and tail
# DataFrame.collect() collects the distributed data to the driver side as the local data in Python. Note that this can throw an out-of-memory error when the dataset is too large to fit in the driver side because it collects all the data from executors to the driver side.
df.collect()


In [ ]:
df.take(5)  # Returns the first n rows as a list of Row.

In [ ]:
df.tail(5)  # Returns the last n rows as a list of Row objects.

In [ ]:
# spark dataframe to pandas
# PySpark DataFrame also provides the conversion back to a pandas DataFrame to leverage pandas API. Note that toPandas also collects all data into the driver side that can easily cause an out-of-memory-error when the data is too large to fit into the driver side.
pdf = df.toPandas()
print(type(pdf))
pdf

## 4. Selecting and Accessing Data


In [ ]:
# PySpark DataFrame is lazily evaluated and simply selecting a column does not trigger the computation but it returns a Column instance.
df.name

In [ ]:
# actually, most of operations on columns return the column instance and not the actual data.
from pyspark.sql import Column
from pyspark.sql.functions import upper

type(df.name), type(df.address), type(upper(df.address))

In [ ]:
# These Columns can be used to select the columns from a DataFrame. For example, DataFrame.select() takes the Column instances that returns another DataFrame.
df.select(df.name, df.address).show()

In [ ]:
# Assign new Column instance.
df.withColumn("upper_address", upper(df.address)).show()  # Add a new column with the upper case of address.

In [ ]:
# Filter data in dataframe
# To select a subset of rows, use DataFrame.filter().
# use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.
df.filter(df.name == 'Lisa Malone').show()  
df.filter(df.name.contains('John')).show()  
df.filter(df.name.startswith("A") & df.name.endswith("son")).show()  # Filter rows where name starts with "A" and ends with "son".

## 5. Applying a function

In [ ]:
# PySpark supports various UDFs and APIs to allow users to execute Python native functions. See also the latest Pandas UDFs and Pandas Function APIs. For instance, the example below allows users to directly use the APIs in a pandas Series within Python native function.
from pyspark.sql.functions import pandas_udf


@pandas_udf("string")
def pandas_append_a(series: pd.Series) -> pd.Series:
    """Append 'a' to each element in the pandas Series."""
    return series + ' a'

df.select(pandas_append_a(df.name)).show()  # Apply the pandas UDF to the 'name' column and show the result.
# this required to install pyarrow in the spark worker docker container (modified the base image via spark.Dockerfile)

In [ ]:
# Another example is DataFrame.mapInPandas which allows users directly use the APIs in a pandas DataFrame without any restrictions such as the result length.

# def pandas_filter_function(pdf: pd.DataFrame) -> pd.DataFrame:
#     """Filter rows in the pandas DataFrame where the 'name' column contains 'John'."""
#     return pdf[pdf['name'].str.contains('John')]

def pandas_filter_function(iterator):
    for pandas_df in iterator:
        # yield pandas_df[pandas_df['name'].str.contains('John')]
        yield pandas_df[pandas_df.name == "Kevin Avila"]  # Filter rows where name is "Kevin Avila".

df.mapInPandas(pandas_filter_function, schema=df.schema).show()  # Apply the pandas function to the DataFrame and show the result.
# mapInPandas lets you apply a function that transforms one Pandas DataFrame into another, operating per-partition, with full control over row filtering/transformation using pandas/numpy operations instead of Spark's built-in functions.

# What actually happens under the hood:

# Partitioning: spark_df is split across partitions (same as any Spark operation). Each partition lives on some executor.
# Per-partition conversion to Pandas: For each partition, Spark converts the partition's rows into one or more Pandas DataFrames — via Arrow, not row-by-row Python serialization, which is why it's fast. It hands your function an iterator of Pandas DataFrames, not a single DataFrame for the whole dataset.
# Your function runs once per batch, per partition.

## 6. Grouping data

In [ ]:
# PySpark DataFrame also provides a way of handling grouped data by using the common approach, split-apply-combine strategy. It groups the data by a certain condition applies a function to each group and then combines them back to the DataFrame.

# Grouping and then applying the avg() function to the resulting groups.
df.groupBy("country").avg("salary").show()

In [ ]:
# You can also apply a Python native function against each group by using pandas API.
def plus_mean(pandas_df: pd.DataFrame) -> pd.DataFrame:
    """Update salary in pandas DataFrame with the sum of 'salary' and the mean of 'salary'."""
    return pandas_df.assign(salary=pandas_df["salary"]+pandas_df["salary"].mean())

df.groupBy("country").applyInPandas(plus_mean, schema=df.schema).show()  # Apply the pandas function to each group and show the result.

In [ ]:
# cogrouping
df1 = spark.createDataFrame(
    [(20000101, 1, 1.0), (20000101, 2, 2.0), (20000102, 1, 3.0), (20000102, 2, 4.0)],
    ('time', 'id', 'v1'))

df2 = spark.createDataFrame(
    [(20000101, 1, 'x'), (20000101, 2, 'y')],
    ('time', 'id', 'v2'))

def merge_ordered(l, r):
    return pd.merge_ordered(l, r)

df1.groupby('id').cogroup(df2.groupby('id')).applyInPandas(
    merge_ordered, schema='time int, id int, v1 double, v2 string').show()

# For id=1, it pairs df1's id=1 rows with df2's id=1 rows. Same for id=2. If a key exists in one side but not the other, you still get a pair — just with an empty pandas DataFrame on the missing side.
# .applyInPandas(merge_ordered, schema=...) runs your function once per key, receiving both sides as separate pandas DataFrames

## 7. Getting data in/out

In [ ]:
# CSV is straightforward and easy to use.
# Parquet and ORC are efficient and compact file formats to read and write faster.
# other data sources available in PySpark such as JDBC, text, binaryFile, Avro, etc

In [ ]:
# csv
# df.write.csv('/data/foo.csv', header=True)
# spark.read.csv('foo.csv', header=True).show()


# df.write.csv('s3a://pyspark-labs/foo.csv', header=True, mode='overwrite')

# df2 = spark.read.csv('s3a://pyspark-labs/foo.csv', header=True)
# df2.show()

# ** IMPORTANT **
# pyspark driver and spark executors both should be pointing to the same s3 using the same url, can't be pyspark-labs-minio:9000(executor) and localhost:9100(driver) at the same time.
# mitigated this by creating a dockerized deployment of this pyspark-app, now the driver and executor both are using pyspark-labs-minio:9000 as url for s3 (minio).

## 7. Working with SQL

In [ ]:
# DataFrame and Spark SQL share the same execution engine so they can be interchangeably used seamlessly. For example, you can register the DataFrame as a table and run a SQL easily as below

In [ ]:
df.createOrReplaceTempView("user_data")

In [ ]:
df.show(n=2)

In [ ]:
spark.sql("select * from user_data where name like 'Joseph Jackson'").show()

In [ ]:
# user defined functions (UDF)
# A UDF (User-Defined Function) is a custom function you write to extend Spark's built-in operations, letting you apply your own Python logic to DataFrame columns or SQL queries.
# In this context specifically, @pandas_udf creates a Pandas UDF (vectorized UDF). Instead of processing one row at a time (like a plain Python UDF), it operates on entire batches of data as pandas Series, using Apache Arrow to transfer data efficiently between the JVM and Python. This makes it much faster than a regular row-at-a-time UDF.
# So the point the text is making: once registered, the same UDF works both in the DataFrame API and directly in SQL strings, with no extra wrapping needed.

@pandas_udf("integer")
def add_one(s: pd.Series) -> pd.Series:
    return s + 1

spark.udf.register("add_one", add_one)
spark.sql("SELECT add_one(salary) FROM user_data where name like 'Joseph Jackson'").show()

In [ ]:
# These SQL expressions can directly be mixed and used as PySpark columns.

from pyspark.sql.functions import expr

df.selectExpr('add_one(salary)').show()
df.select(expr('count(*)')).show() # 100 rows
df.select(expr('count(*)') > 10).show() # true